# Badge 1: Data Warehousing Workshop: Lesson 2: Identity & Access

Databases, ownership, roles, and the warehouse ownership challenge.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `badge1_dora_checks.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `00_badge1_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH

In [ ]:
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🤔 UI Changed? Of course it has!

Reading and a knowledge check only.

## Page: 🥋 Create a Database

> **Course:** UI steps (screenshot DWW_006): Catalog → Database Explorer → **+ Database** → Name **DEMO_DB** → **Create**. Dialog shows *Owner: ACCOUNTADMIN*.

In [ ]:
-- SQL equivalent of the UI steps, run as the ACCOUNTADMIN stand-in
use role {{ADMIN_ROLE}};
create database DEMO_DB;

## Page: 🥋 Explore Your New Database

> **Course:** Explore the database you created (DWW_011). How many schemas are included? What are their names? What is inside each schema?

In [ ]:
show schemas in database DEMO_DB;

In [ ]:
show databases like 'DEMO_DB';

## Page: 🥋 Transfer Ownership of Your New Database

> **Course:** UI steps (DWW_013, DWW_014): DEMO_DB → **…** → **Transfer Ownership** → Transfer to **SYSADMIN** → **Transfer**.

In [ ]:
-- SQL equivalent of Transfer Ownership
grant ownership on database DEMO_DB to role {{SYSADMIN_ROLE}};
show databases like 'DEMO_DB';

> **Course:** 🥋 Switch Your System Role Back to SYSADMIN (DWW_008).

In [ ]:
use role {{SYSADMIN_ROLE}};
select current_role();

## Page: 🎭 Identity vs. Access

Knowledge checks only.

## Page: 🥋 Changing Your Role Setting

> **Course:** UI steps (DWW_016): Governance & security → Users & roles → **Roles** → **Graph**; click each role name and read what each role does.

In [ ]:
-- SQL equivalent of the Roles graph for the isolated roles
use role {{ADMIN_ROLE}};
show grants to role {{ADMIN_ROLE}};

In [ ]:
show grants of role {{SYSADMIN_ROLE}};

## Page: 📓 RBAC Review / 🎭 DAC Model & Default Role

> **Course:** Because of the combination of RBAC and DAC in Snowflake, when we create something, the ROLE we were using at the time we created it is the role that OWNS it.

In [ ]:
-- DEMO_DB now belongs to SYSADMIN, but its PUBLIC schema still belongs to the creating role
show schemas in database DEMO_DB;

## Page: 📓 Roles and Warehouse Visibility

> **Course:** Three warehouses were created for you (DWW_022): COMPUTE_WH (owned by ACCOUNTADMIN), a learning/sandbox warehouse, and SYSTEM$STREAMLIT_NOTEBOOK_WH.

> ⚠️ **Audit note:** The page contains two copies of this text that disagree: one says the second warehouse is **SANDBOX_WH**, the other **SNOWFLAKE_LEARNING_WH**. Screenshot DWW_022 shows SANDBOX_WH.

In [ ]:
use role {{SYSADMIN_ROLE}};
show warehouses;

In [ ]:
use role {{ADMIN_ROLE}};
show warehouses;

> **Course:** 🎯 CHALLENGE LAB: Make the Warehouse Available to SYSADMIN. Transfer the COMPUTE_WH warehouse to SYSADMIN. (UI, DWW_023: Compute → Warehouses → COMPUTE_WH → … → Transfer Ownership → SYSADMIN → Transfer.)

In [ ]:
-- Derived solution (SQL equivalent of the UI transfer)
grant ownership on warehouse {{WAREHOUSE}} to role {{SYSADMIN_ROLE}} copy current grants;
use role {{SYSADMIN_ROLE}};
show warehouses;

## Page: 🏁 Summary & Next Steps

> **Course:** Are you ready for Lesson 3? Can see the COMPUTE_WH warehouse while in the SYSADMIN role · Can see your DEMO_DB database when SYSADMIN · Can see your UTIL_DB database when SYSADMIN.

In [ ]:
use role {{SYSADMIN_ROLE}};
show databases;

In [ ]:
names = set(ready_check["name"])
for db in ["DEMO_DB", "UTIL_DB"]:
    print(f"{db} visible to SYSADMIN stand-in:", db in names)

> ⚠️ **Audit note:** This checklist expects **UTIL_DB**, which the course has not told the learner to create (see `00_badge1_setup`).